In [1]:
%pip install pandas numby matplotlib seaborn scikit-learn geopy

Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from geopy.distance import geodesic
import warnings
warnings.filterwarnings('ignore')

class SinkholeDataPreprocessor:
    """
    싱크홀 위험도 예측을 위한 CSV 데이터 전처리 클래스
    """
    
    def __init__(self, grid_size_km=0.5):
        self.grid_size_km = grid_size_km
        
    def load_csv_data(self):
        """CSV 파일들 로드"""
        # 실제 파일명으로 수정 필요
        construction_df = pd.read_csv('data/공사현황_정제.csv', encoding='utf-8')
        rainfall_df = pd.read_csv('data/지역구별누적강수량.csv', encoding='utf-8')
        pothole_df = pd.read_csv('data/포트홀및싱크홀정보.csv', encoding='utf-8')
        population_df = pd.read_csv('data/유동인구수.csv', encoding='utf-8')
        subway_df = pd.read_csv('data/지하철유동인구수.csv', encoding='utf-8')
        
        return construction_df, rainfall_df, pothole_df, population_df, subway_df
    
    def clean_construction_data(self, df):
        """공사현황 데이터 정제"""
        # 날짜 컬럼 변환
        df['착공일'] = pd.to_datetime(df['착공일'], errors='coerce')
        df['준공일'] = pd.to_datetime(df['준공일'], errors='coerce')
        
        # 위도, 경도 정제 (좌표상태가 'API성공'인 것만)
        df = df[df['좌표상태'].notna()].copy()
        df['위도'] = pd.to_numeric(df['위도'], errors='coerce')
        df['경도'] = pd.to_numeric(df['경도'], errors='coerce')
        
        # 유효한 좌표만 필터링 (서울시 범위)
        df = df[
            (df['위도'].between(37.4, 37.7)) & 
            (df['경도'].between(126.8, 127.2))
        ].copy()
        
        # 위험도 레벨 정제
        df['현재_위험도_레벨'] = pd.to_numeric(df['현재_위험도_레벨'], errors='coerce').fillna(0)
        df['기본_위험도_레벨'] = pd.to_numeric(df['기본_위험도_레벨'], errors='coerce').fillna(0)
        
        print(f"공사현황 데이터 정제 완료: {len(df)}건")
        return df
    
    def clean_rainfall_data(self, df):
        """강수량 데이터 정제"""
        # 날짜 컬럼 변환
        df['날짜'] = pd.to_datetime(df['날짜'].astype(str), format='%Y%m%d')
        df['누적강수량'] = pd.to_numeric(df['누적강수량'], errors='coerce').fillna(0)
        
        # 위도, 경도 정제
        df['위도'] = pd.to_numeric(df['위도'], errors='coerce')
        df['경도'] = pd.to_numeric(df['경도'], errors='coerce')
        
        print(f"강수량 데이터 정제 완료: {len(df)}건")
        return df
    
    def clean_pothole_data(self, df):
        """포트홀/싱크홀 데이터 정제"""
        # 날짜 컬럼 변환
        df['날짜'] = pd.to_datetime(df['날짜'].astype(str), format='%Y%m%d')
        
        # 위도, 경도 정제
        df['위도'] = pd.to_numeric(df['위도'], errors='coerce')
        df['경도'] = pd.to_numeric(df['경도'], errors='coerce')
        
        # 유효한 좌표만 필터링
        df = df[
            (df['위도'].between(37.4, 37.7)) & 
            (df['경도'].between(126.8, 127.2))
        ].copy()
        
        # 피해규모점수 정제
        df['피해규모점수'] = pd.to_numeric(df['피해규모점수'], errors='coerce').fillna(0)
        
        print(f"포트홀/싱크홀 데이터 정제 완료: {len(df)}건")
        return df
    
    def clean_population_data(self, df):
        """유동인구 데이터 정제"""
        # 날짜 컬럼 변환
        df['날짜'] = pd.to_datetime(df['날짜'].astype(str), format='%Y%m%d')
        
        # 위도, 경도 정제
        df['위도'] = pd.to_numeric(df['위도'], errors='coerce')
        df['경도'] = pd.to_numeric(df['경도'], errors='coerce')
        
        # 방문자수 정제
        df['방문자수'] = pd.to_numeric(df['방문자수'], errors='coerce').fillna(0)
        
        print(f"유동인구 데이터 정제 완료: {len(df)}건")
        return df
    
    def clean_subway_data(self, df):
        """지하철 유동인구 데이터 정제"""
        # 날짜 컬럼 변환
        df['날짜'] = pd.to_datetime(df['날짜'].astype(str), format='%Y%m%d')
        
        # 위도, 경도 정제
        df['위도'] = pd.to_numeric(df['위도'], errors='coerce')
        df['경도'] = pd.to_numeric(df['경도'], errors='coerce')
        
        # 승하차 승객수 정제
        df['승하차총승객수'] = pd.to_numeric(df['승하차총승객수'], errors='coerce').fillna(0)
        
        print(f"지하철 유동인구 데이터 정제 완료: {len(df)}건")
        return df
    
    def create_seoul_grid(self):
        """서울시 격자 시스템 생성"""
        # 서울시 경계 좌표
        min_lat, max_lat = 37.4, 37.7
        min_lng, max_lng = 126.8, 127.2
        
        # 격자 크기 계산 (위도 1도 ≈ 111km, 경도 1도 ≈ 88km)
        lat_step = self.grid_size_km / 111
        lng_step = self.grid_size_km / 88
        
        grids = []
        grid_id = 0
        
        lat = min_lat
        while lat < max_lat:
            lng = min_lng
            while lng < max_lng:
                grids.append({
                    'grid_id': f'G_{grid_id:04d}',
                    'center_lat': round(lat + lat_step/2, 6),
                    'center_lng': round(lng + lng_step/2, 6),
                    'min_lat': lat,
                    'max_lat': lat + lat_step,
                    'min_lng': lng,
                    'max_lng': lng + lng_step
                })
                lng += lng_step
                grid_id += 1
            lat += lat_step
        
        grid_df = pd.DataFrame(grids)
        print(f"격자 시스템 생성 완료: {len(grid_df)}개 격자")
        return grid_df
    
    def assign_to_grid(self, df, grid_df):
        """데이터포인트를 격자에 할당"""
        def find_grid_id(lat, lng):
            if pd.isna(lat) or pd.isna(lng):
                return None
            
            for _, grid in grid_df.iterrows():
                if (grid['min_lat'] <= lat < grid['max_lat'] and 
                    grid['min_lng'] <= lng < grid['max_lng']):
                    return grid['grid_id']
            return None
        
        df['grid_id'] = df.apply(lambda x: find_grid_id(x['위도'], x['경도']), axis=1)
        return df[df['grid_id'].notna()].copy()
    
    def aggregate_construction_features(self, construction_df, grid_df, base_date):
        """격자별 공사 특성 집계"""
        features = []
        
        for _, grid in grid_df.iterrows():
            grid_constructions = construction_df[
                construction_df['grid_id'] == grid['grid_id']
            ].copy()
            
            if len(grid_constructions) == 0:
                feature = {
                    'grid_id': grid['grid_id'],
                    'active_construction_count': 0,
                    'completed_construction_count': 0,
                    'construction_density': 0,
                    'avg_construction_risk': 0,
                    'road_construction_ratio': 0,
                    'utility_construction_ratio': 0,
                    'construction_duration_avg': 0
                }
            else:
                # 활성/완료 공사 분류
                active_count = len(grid_constructions[
                    grid_constructions['공사상태'].isin(['진행중', '시작전'])
                ])
                completed_count = len(grid_constructions[
                    grid_constructions['공사상태'] == '완료'
                ])
                
                # 공사 유형별 비율
                total_count = len(grid_constructions)
                road_count = len(grid_constructions[
                    grid_constructions['공사유형'].str.contains('도로|포장', na=False)
                ])
                utility_count = len(grid_constructions[
                    grid_constructions['공사유형'].str.contains('전력|상수|하수|가스', na=False)
                ])
                
                # 공사 기간 계산
                duration_days = []
                for _, row in grid_constructions.iterrows():
                    if pd.notna(row['착공일']) and pd.notna(row['준공일']):
                        duration = (row['준공일'] - row['착공일']).days
                        if duration > 0:
                            duration_days.append(duration)
                
                feature = {
                    'grid_id': grid['grid_id'],
                    'active_construction_count': active_count,
                    'completed_construction_count': completed_count,
                    'construction_density': total_count / (self.grid_size_km ** 2),
                    'avg_construction_risk': grid_constructions['현재_위험도_레벨'].mean(),
                    'road_construction_ratio': road_count / total_count if total_count > 0 else 0,
                    'utility_construction_ratio': utility_count / total_count if total_count > 0 else 0,
                    'construction_duration_avg': np.mean(duration_days) if duration_days else 0
                }
            
            features.append(feature)
        
        return pd.DataFrame(features)
    
    def aggregate_weather_features(self, rainfall_df, grid_df, base_date):
        """격자별 기상 특성 집계"""
        features = []
        
        # 구별 강수량을 격자에 매핑 (각 격자는 가장 가까운 구의 강수량 사용)
        district_coords = {
            '강남구': (37.5173, 127.0473),
            '강동구': (37.5301, 127.1238),
            '강북구': (37.6396, 127.0255),
            '강서구': (37.5509, 126.8495),
            '관악구': (37.4781, 126.9515),
            # ... 나머지 구 추가 필요
        }
        
        for _, grid in grid_df.iterrows():
            grid_lat, grid_lng = grid['center_lat'], grid['center_lng']
            
            # 가장 가까운 구 찾기
            closest_district = None
            min_distance = float('inf')
            
            for district, (lat, lng) in district_coords.items():
                distance = geodesic((grid_lat, grid_lng), (lat, lng)).kilometers
                if distance < min_distance:
                    min_distance = distance
                    closest_district = district
            
            # 해당 구의 강수량 데이터 가져오기
            district_rainfall = rainfall_df[
                rainfall_df['지역구'] == closest_district
            ].copy()
            
            # 기간별 강수량 계산
            rainfall_1day = 0
            rainfall_3day = 0
            rainfall_7day = 0
            rainfall_30day = 0
            heavy_rain_days = 0
            
            for i in range(30):
                check_date = base_date - timedelta(days=i)
                day_data = district_rainfall[
                    district_rainfall['날짜'] == check_date
                ]
                
                if not day_data.empty:
                    daily_rain = day_data['누적강수량'].iloc[0]
                    
                    if i == 0:
                        rainfall_1day = daily_rain
                    if i < 3:
                        rainfall_3day += daily_rain
                    if i < 7:
                        rainfall_7day += daily_rain
                    if i < 30:
                        rainfall_30day += daily_rain
                        if daily_rain >= 50:  # 집중호우 기준
                            heavy_rain_days += 1
            
            feature = {
                'grid_id': grid['grid_id'],
                'closest_district': closest_district,
                'rainfall_1day': rainfall_1day,
                'rainfall_3day': rainfall_3day,
                'rainfall_7day': rainfall_7day,
                'rainfall_30day': rainfall_30day,
                'heavy_rain_days': heavy_rain_days
            }
            
            features.append(feature)
        
        return pd.DataFrame(features)
    
    def aggregate_incident_features(self, pothole_df, grid_df):
        """격자별 과거 사고 특성 집계"""
        features = []
        
        for _, grid in grid_df.iterrows():
            grid_lat, grid_lng = grid['center_lat'], grid['center_lng']
            
            # 거리별 사고 계산
            incidents_500m = []
            incidents_1km = []
            
            for _, incident in pothole_df.iterrows():
                try:
                    distance = geodesic(
                        (grid_lat, grid_lng),
                        (incident['위도'], incident['경도'])
                    ).kilometers
                    
                    if distance <= 0.5:
                        incidents_500m.append(incident)
                    if distance <= 1.0:
                        incidents_1km.append(incident)
                except:
                    continue
            
            # 사고 원인 다양성 계산
            if incidents_1km:
                causes = [inc['발생원인구분'] for inc in incidents_1km if pd.notna(inc['발생원인구분'])]
                cause_diversity = len(set(causes)) if causes else 0
            else:
                cause_diversity = 0
            
            feature = {
                'grid_id': grid['grid_id'],
                'incident_count_500m': len(incidents_500m),
                'incident_count_1km': len(incidents_1km),
                'avg_damage_score': np.mean([inc['피해규모점수'] for inc in incidents_1km]) if incidents_1km else 0,
                'max_damage_score': np.max([inc['피해규모점수'] for inc in incidents_1km]) if incidents_1km else 0,
                'incident_cause_diversity': cause_diversity
            }
            
            features.append(feature)
        
        return pd.DataFrame(features)
    
    def aggregate_population_features(self, population_df, subway_df, grid_df):
        """격자별 인구 밀도 특성 집계"""
        features = []
        
        for _, grid in grid_df.iterrows():
            # 해당 격자의 유동인구 데이터
            grid_population = population_df[
                population_df['grid_id'] == grid['grid_id']
            ]
            
            grid_subway = subway_df[
                subway_df['grid_id'] == grid['grid_id']
            ]
            
            # 유동인구 통계
            if len(grid_population) > 0:
                visitor_density = grid_population['방문자수'].sum() / (self.grid_size_km ** 2)
                population_variation = grid_population['방문자수'].std()
                peak_ratio = (grid_population['방문자수'].max() / 
                            (grid_population['방문자수'].min() + 1))
            else:
                visitor_density = 0
                population_variation = 0
                peak_ratio = 1
            
            # 지하철 승객 밀도
            if len(grid_subway) > 0:
                subway_density = grid_subway['승하차총승객수'].sum() / (self.grid_size_km ** 2)
            else:
                subway_density = 0
            
            feature = {
                'grid_id': grid['grid_id'],
                'visitor_density': visitor_density,
                'subway_passenger_density': subway_density,
                'population_variation': population_variation,
                'peak_population_ratio': peak_ratio
            }
            
            features.append(feature)
        
        return pd.DataFrame(features)
    
    def create_target_variable(self, construction_df, pothole_df, grid_df):
        """타겟 변수 생성"""
        targets = []
        
        for _, grid in grid_df.iterrows():
            # 실제 사고 발생 여부
            grid_incidents = pothole_df[pothole_df['grid_id'] == grid['grid_id']]
            has_incident = 1 if len(grid_incidents) > 0 else 0
            
            # 종합 위험도 계산
            risk_score = 0
            
            # 1. 공사 위험도 (40%)
            grid_constructions = construction_df[construction_df['grid_id'] == grid['grid_id']]
            if len(grid_constructions) > 0:
                construction_risk = grid_constructions['현재_위험도_레벨'].mean()
                risk_score += construction_risk * 0.4
            
            # 2. 과거 사고 이력 (40%)
            if len(grid_incidents) > 0:
                incident_risk = grid_incidents['피해규모점수'].max()
                risk_score += incident_risk * 0.4
            
            # 3. 활성 공사 수 (20%)
            active_constructions = len(grid_constructions[
                grid_constructions['공사상태'].isin(['진행중', '시작전'])
            ]) if len(grid_constructions) > 0 else 0
            risk_score += min(active_constructions, 4) * 0.2
            
            # 위험도 레벨로 변환 (0-4)
            if risk_score == 0:
                risk_level = 0
            elif risk_score <= 1:
                risk_level = 1
            elif risk_score <= 2:
                risk_level = 2
            elif risk_score <= 3:
                risk_level = 3
            else:
                risk_level = 4
            
            targets.append({
                'grid_id': grid['grid_id'],
                'risk_level': risk_level,
                'has_incident': has_incident,
                'risk_score': round(risk_score, 3)
            })
        
        return pd.DataFrame(targets)
    
    def process_all_data(self, base_date=None):
        """전체 데이터 처리 파이프라인"""
        if base_date is None:
            base_date = datetime(2024, 6, 25)
        
        print("=== 싱크홀 위험도 예측 데이터 전처리 시작 ===")
        
        # 1. 데이터 로드
        print("\n1. CSV 데이터 로드 중...")
        construction_df, rainfall_df, pothole_df, population_df, subway_df = self.load_csv_data()
        
        # 2. 데이터 정제
        print("\n2. 데이터 정제 중...")
        construction_df = self.clean_construction_data(construction_df)
        rainfall_df = self.clean_rainfall_data(rainfall_df)
        pothole_df = self.clean_pothole_data(pothole_df)
        population_df = self.clean_population_data(population_df)
        subway_df = self.clean_subway_data(subway_df)
        
        # 3. 격자 시스템 생성
        print("\n3. 격자 시스템 생성 중...")
        grid_df = self.create_seoul_grid()
        
        # 4. 데이터를 격자에 할당
        print("\n4. 데이터를 격자에 할당 중...")
        print("\n4.1 counstruction데이터를 격자에 할당 중...")
        construction_df = self.assign_to_grid(construction_df, grid_df)
        print("\n4.2 포트홀 데이터 격자에 할당 중...")
        pothole_df = self.assign_to_grid(pothole_df, grid_df)
        print("\n4.3 인구 데이터 격자에 할당 중...")
        population_df = self.assign_to_grid(population_df, grid_df)
        print("\n4.4 지하철 데이터 격자에 할당 중...")
        subway_df = self.assign_to_grid(subway_df, grid_df)
        
        # 5. 특성 집계
        print("\n5. 격자별 특성 집계 중...")
        construction_features = self.aggregate_construction_features(construction_df, grid_df, base_date)
        weather_features = self.aggregate_weather_features(rainfall_df, grid_df, base_date)
        incident_features = self.aggregate_incident_features(pothole_df, grid_df)
        population_features = self.aggregate_population_features(population_df, subway_df, grid_df)
        target_variables = self.create_target_variable(construction_df, pothole_df, grid_df)
        
        # 6. 모든 특성 통합
        print("\n6. 최종 데이터셋 생성 중...")
        final_df = grid_df[['grid_id', 'center_lat', 'center_lng']].copy()
        
        # 모든 특성 병합
        for features in [construction_features, weather_features, 
                        incident_features, population_features, target_variables]:
            final_df = final_df.merge(features, on='grid_id', how='left')
        
        # 7. 시간적 특성 추가
        final_df['date'] = base_date
        final_df['month'] = base_date.month
        final_df['quarter'] = (base_date.month - 1) // 3 + 1
        final_df['is_rainy_season'] = 1 if base_date.month in [6, 7, 8] else 0
        final_df['day_of_year'] = base_date.timetuple().tm_yday
        
        # 8. 결측치 처리
        final_df = final_df.fillna(0)
        
        # 9. 파생 특성 생성
        final_df['construction_weather_interaction'] = (
            final_df['construction_density'] * final_df['rainfall_7day']
        )
        final_df['population_construction_ratio'] = (
            final_df['visitor_density'] / (final_df['construction_density'] + 1)
        )
        final_df['vulnerability_index'] = (
            final_df['avg_construction_risk'] * 0.3 +
            final_df['max_damage_score'] * 0.4 +
            final_df['heavy_rain_days'] * 0.3
        )
        
        print(f"\n=== 전처리 완료 ===")
        print(f"최종 데이터셋 크기: {final_df.shape}")
        print(f"특성 수: {final_df.shape[1] - 4}")  # grid_id, lat, lng, date 제외
        print(f"위험도 분포:")
        print(final_df['risk_level'].value_counts().sort_index())
        
        return final_df
    
    def save_processed_data(self, final_df, output_path='sinkhole_processed_data.csv'):
        """처리된 데이터 저장"""
        final_df.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"처리된 데이터가 {output_path}에 저장되었습니다.")
        
        # 데이터 요약 정보도 저장
        summary_info = {
            'total_grids': len(final_df),
            'feature_count': final_df.shape[1] - 4,
            'risk_distribution': final_df['risk_level'].value_counts().to_dict(),
            'incident_rate': final_df['has_incident'].mean(),
            'date_processed': datetime.now().isoformat()
        }
        
        import json
        with open('data_summary.json', 'w', encoding='utf-8') as f:
            json.dump(summary_info, f, ensure_ascii=False, indent=2)

# 실행 예시
if __name__ == "__main__":
    # 전처리 실행
    processor = SinkholeDataPreprocessor(grid_size_km=0.5)
    
    try:
        # 전체 데이터 처리
        final_dataset = processor.process_all_data(base_date=datetime(2024, 6, 25))
        
        # 결과 저장
        processor.save_processed_data(final_dataset)
        
        # 기본 통계 출력
        print("\n=== 데이터 기본 통계 ===")
        print(final_dataset.describe())
        
    except Exception as e:
        print(f"오류 발생: {e}")
        print("CSV 파일 경로와 컬럼명을 확인해주세요.")

=== 싱크홀 위험도 예측 데이터 전처리 시작 ===

1. CSV 데이터 로드 중...

2. 데이터 정제 중...
공사현황 데이터 정제 완료: 389건
강수량 데이터 정제 완료: 64925건
포트홀/싱크홀 데이터 정제 완료: 20451건
유동인구 데이터 정제 완료: 18880건
지하철 유동인구 데이터 정제 완료: 741755건

3. 격자 시스템 생성 중...
격자 시스템 생성 완료: 4757개 격자

4. 데이터를 격자에 할당 중...

4.1 counstruction데이터를 격자에 할당 중...

4.2 포트홀 데이터 격자에 할당 중...


KeyboardInterrupt: 

In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from geopy.distance import geodesic
import warnings
warnings.filterwarnings('ignore')

# 병렬 처리 라이브러리
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from multiprocessing import Pool, cpu_count
import threading
from functools import partial
import time

class OptimizedSinkholeDataPreprocessor:
    """
    병렬 처리 최적화된 싱크홀 위험도 예측 데이터 전처리 클래스
    """
    
    def __init__(self, grid_size_km=0.5, n_processes=None, n_threads=None):
        self.grid_size_km = grid_size_km
        self.n_processes = n_processes or max(1, cpu_count() - 1)  # CPU 코어 수 - 1
        self.n_threads = n_threads or min(32, (cpu_count() or 1) + 4)  # I/O 작업용
        
        print(f"병렬 처리 설정: {self.n_processes}개 프로세스, {self.n_threads}개 스레드")
        
    def load_csv_data_parallel(self):
        """병렬로 CSV 파일들 로드"""
        file_configs = [
            ('data/공사현황정제.csv', 'construction'),
            ('data/지역구별누적강수량.csv', 'rainfall'),
            ('data/포트홀및싱크홀정보.csv', 'pothole'),
            ('data/유동인구수.csv', 'population'),
            ('data/지하철유동인구수.csv', 'subway')
        ]
        
        def load_single_file(config):
            filename, key = config
            try:
                start_time = time.time()
                df = pd.read_csv(filename, encoding='utf-8')
                load_time = time.time() - start_time
                print(f"✓ {key} 데이터 로드 완료: {len(df)}건 ({load_time:.2f}초)")
                return key, df
            except Exception as e:
                print(f"✗ {key} 데이터 로드 실패: {e}")
                return key, pd.DataFrame()
        
        print("병렬 CSV 로드 시작...")
        with ThreadPoolExecutor(max_workers=self.n_threads) as executor:
            future_to_config = {executor.submit(load_single_file, config): config for config in file_configs}
            results = {}
            
            for future in as_completed(future_to_config):
                key, df = future.result()
                results[key] = df
        
        return (results['construction'], results['rainfall'], results['pothole'], 
                results['population'], results['subway'])
    
    def clean_data_parallel(self, dataframes):
        """병렬로 데이터 정제 (ThreadPoolExecutor 사용으로 변경)"""
        construction_df, rainfall_df, pothole_df, population_df, subway_df = dataframes
        
        cleaning_tasks = [
            (construction_df, 'construction'),
            (rainfall_df, 'rainfall'),
            (pothole_df, 'pothole'),
            (population_df, 'population'),
            (subway_df, 'subway')
        ]
        
        def clean_single_dataframe(task):
            df, name = task
            start_time = time.time()
            
            # 각 데이터 타입별로 적절한 정제 함수 호출
            if name == 'construction':
                cleaned_df = self.clean_construction_data(df)
            elif name == 'rainfall':
                cleaned_df = self.clean_rainfall_data(df)
            elif name == 'pothole':
                cleaned_df = self.clean_pothole_data(df)
            elif name == 'population':
                cleaned_df = self.clean_population_data(df)
            elif name == 'subway':
                cleaned_df = self.clean_subway_data(df)
            else:
                cleaned_df = df
            
            clean_time = time.time() - start_time
            print(f"✓ {name} 데이터 정제 완료: {len(cleaned_df)}건 ({clean_time:.2f}초)")
            return name, cleaned_df
        
        print("병렬 데이터 정제 시작...")
        # ProcessPoolExecutor 대신 ThreadPoolExecutor 사용 (pickle 문제 해결)
        with ThreadPoolExecutor(max_workers=min(5, self.n_threads)) as executor:
            future_to_task = {executor.submit(clean_single_dataframe, task): task for task in cleaning_tasks}
            results = {}
            
            for future in as_completed(future_to_task):
                name, cleaned_df = future.result()
                results[name] = cleaned_df
        
        return (results['construction'], results['rainfall'], results['pothole'],
                results['population'], results['subway'])
    
    def clean_construction_data(self, df):
        """공사현황 데이터 정제 (정적 메서드로 병렬 처리 가능)"""
        df = df.copy()
        
        # 날짜 컬럼 변환
        df['착공일'] = pd.to_datetime(df['착공일'], errors='coerce')
        df['준공일'] = pd.to_datetime(df['준공일'], errors='coerce')
        
        # 위도, 경도 정제
        df = df[df['좌표상태'].notna()].copy()
        df['위도'] = pd.to_numeric(df['위도'], errors='coerce')
        df['경도'] = pd.to_numeric(df['경도'], errors='coerce')
        
        # 서울시 범위 필터링
        df = df[
            (df['위도'].between(37.4, 37.7)) & 
            (df['경도'].between(126.8, 127.2))
        ].copy()
        
        # 위험도 레벨 정제
        df['현재_위험도_레벨'] = pd.to_numeric(df['현재_위험도_레벨'], errors='coerce').fillna(0)
        df['기본_위험도_레벨'] = pd.to_numeric(df['기본_위험도_레벨'], errors='coerce').fillna(0)
        
        return df
    
    def clean_rainfall_data(self, df):
        """강수량 데이터 정제"""
        df = df.copy()
        df['날짜'] = pd.to_datetime(df['날짜'].astype(str), format='%Y%m%d')
        df['누적강수량'] = pd.to_numeric(df['누적강수량'], errors='coerce').fillna(0)
        df['위도'] = pd.to_numeric(df['위도'], errors='coerce')
        df['경도'] = pd.to_numeric(df['경도'], errors='coerce')
        return df
    
    def clean_pothole_data(self, df):
        """포트홀/싱크홀 데이터 정제"""
        df = df.copy()
        df['날짜'] = pd.to_datetime(df['날짜'].astype(str), format='%Y%m%d')
        df['위도'] = pd.to_numeric(df['위도'], errors='coerce')
        df['경도'] = pd.to_numeric(df['경도'], errors='coerce')
        
        # 서울시 범위 필터링
        df = df[
            (df['위도'].between(37.4, 37.7)) & 
            (df['경도'].between(126.8, 127.2))
        ].copy()
        
        df['피해규모점수'] = pd.to_numeric(df['피해규모점수'], errors='coerce').fillna(0)
        return df
    
    def clean_population_data(self, df):
        """유동인구 데이터 정제"""
        df = df.copy()
        df['날짜'] = pd.to_datetime(df['날짜'].astype(str), format='%Y%m%d')
        df['위도'] = pd.to_numeric(df['위도'], errors='coerce')
        df['경도'] = pd.to_numeric(df['경도'], errors='coerce')
        df['방문자수'] = pd.to_numeric(df['방문자수'], errors='coerce').fillna(0)
        return df
    
    def clean_subway_data(self, df):
        """지하철 유동인구 데이터 정제"""
        df = df.copy()
        df['날짜'] = pd.to_datetime(df['날짜'].astype(str), format='%Y%m%d')
        df['위도'] = pd.to_numeric(df['위도'], errors='coerce')
        df['경도'] = pd.to_numeric(df['경도'], errors='coerce')
        df['승하차총승객수'] = pd.to_numeric(df['승하차총승객수'], errors='coerce').fillna(0)
        return df
    
    def create_seoul_grid(self):
        """서울시 격자 시스템 생성 (벡터화 최적화)"""
        min_lat, max_lat = 37.4, 37.7
        min_lng, max_lng = 126.8, 127.2
        
        lat_step = self.grid_size_km / 111
        lng_step = self.grid_size_km / 88
        
        # 벡터화된 격자 생성
        lat_range = np.arange(min_lat, max_lat, lat_step)
        lng_range = np.arange(min_lng, max_lng, lng_step)
        
        lat_grid, lng_grid = np.meshgrid(lat_range, lng_range, indexing='ij')
        
        # 격자 데이터프레임 생성
        total_grids = lat_grid.size
        grid_df = pd.DataFrame({
            'grid_id': [f'G_{i:04d}' for i in range(total_grids)],
            'center_lat': np.round(lat_grid.flatten() + lat_step/2, 6),
            'center_lng': np.round(lng_grid.flatten() + lng_step/2, 6),
            'min_lat': lat_grid.flatten(),
            'max_lat': lat_grid.flatten() + lat_step,
            'min_lng': lng_grid.flatten(),
            'max_lng': lng_grid.flatten() + lng_step
        })
        
        print(f"격자 시스템 생성 완료: {len(grid_df)}개 격자")
        return grid_df
    
    def assign_to_grid_vectorized(self, df, grid_df):
        """벡터화된 격자 할당 (대폭 성능 개선)"""
        if len(df) == 0 or 'grid_id' in df.columns:
            return df
            
        # 유효한 좌표만 필터링
        valid_coords = df[['위도', '경도']].dropna()
        if len(valid_coords) == 0:
            df['grid_id'] = None
            return df
        
        # 벡터화된 격자 할당
        lat_step = self.grid_size_km / 111
        lng_step = self.grid_size_km / 88
        
        # 격자 인덱스 계산
        lat_idx = ((valid_coords['위도'] - 37.4) / lat_step).astype(int)
        lng_idx = ((valid_coords['경도'] - 126.8) / lng_step).astype(int)
        
        # 범위 내 체크
        lat_max_idx = int((37.7 - 37.4) / lat_step)
        lng_max_idx = int((127.2 - 126.8) / lng_step)
        
        valid_mask = (
            (lat_idx >= 0) & (lat_idx < lat_max_idx) & 
            (lng_idx >= 0) & (lng_idx < lng_max_idx)
        )
        
        # 격자 ID 생성
        grid_indices = lat_idx * lng_max_idx + lng_idx
        grid_ids = ['G_{:04d}'.format(idx) if valid else None 
                   for idx, valid in zip(grid_indices, valid_mask)]
        
        # 원본 데이터프레임에 격자 ID 할당
        df = df.copy()
        df['grid_id'] = None
        df.loc[valid_coords.index, 'grid_id'] = grid_ids
        
        return df[df['grid_id'].notna()].copy()
    
    def assign_all_to_grid_parallel(self, dataframes, grid_df):
        """모든 데이터를 병렬로 격자에 할당"""
        construction_df, rainfall_df, pothole_df, population_df, subway_df = dataframes
        
        assignment_tasks = [
            (construction_df, 'construction'),
            (pothole_df, 'pothole'), 
            (population_df, 'population'),
            (subway_df, 'subway')
        ]
        
        def assign_single_df(task):
            df, name = task
            start_time = time.time()
            assigned_df = self.assign_to_grid_vectorized(df, grid_df)
            assign_time = time.time() - start_time
            print(f"✓ {name} 격자 할당 완료: {len(assigned_df)}건 ({assign_time:.2f}초)")
            return name, assigned_df
        
        print("병렬 격자 할당 시작...")
        with ThreadPoolExecutor(max_workers=self.n_threads) as executor:
            future_to_task = {executor.submit(assign_single_df, task): task for task in assignment_tasks}
            results = {}
            
            for future in as_completed(future_to_task):
                name, assigned_df = future.result()
                results[name] = assigned_df
        
        # rainfall_df는 격자 할당 없이 그대로 반환 (구별 데이터)
        return (results['construction'], rainfall_df, results['pothole'],
                results['population'], results['subway'])
    
    def aggregate_features_parallel(self, dataframes, grid_df, base_date):
        """병렬로 모든 특성 집계 (ThreadPoolExecutor 사용으로 변경)"""
        construction_df, rainfall_df, pothole_df, population_df, subway_df = dataframes
        
        def aggregate_construction_task():
            print("   공사 특성 집계 시작...")
            result = self.aggregate_construction_features_optimized(construction_df, grid_df, base_date)
            print(f"   공사 특성 집계 완료: {len(result)}개 격자")
            return 'construction', result
        
        def aggregate_weather_task():
            return 'weather', self.aggregate_weather_features_optimized(rainfall_df, grid_df, base_date)
        
        def aggregate_incident_task():
            return 'incident', self.aggregate_incident_features_optimized(pothole_df, grid_df)
        
        def aggregate_population_task():
            print("   인구 특성 집계 시작...")
            result = self.aggregate_population_features_optimized((population_df, subway_df), grid_df)
            print(f"   인구 특성 집계 완료: {len(result)}개 격자")
            return 'population', result
        
        def aggregate_target_task():
            print("   타겟 변수 생성 시작...")
            result = self.create_target_variable_optimized((construction_df, pothole_df), grid_df)
            print(f"   타겟 변수 생성 완료: {len(result)}개 격자")
            return 'target', result
        
        aggregation_tasks = [
            aggregate_construction_task,
            aggregate_weather_task,
            aggregate_incident_task,
            aggregate_population_task,
            aggregate_target_task
        ]
        
        print("병렬 특성 집계 시작...")
        # ThreadPoolExecutor 사용으로 변경
        with ThreadPoolExecutor(max_workers=min(5, self.n_threads)) as executor:
            future_to_task = {executor.submit(task): task for task in aggregation_tasks}
            results = {}
            
            for future in as_completed(future_to_task):
                start_time = time.time()
                name, result = future.result()
                agg_time = time.time() - start_time
                print(f"✓ {name} 특성 집계 완료 ({agg_time:.2f}초)")
                results[name] = result
        
        return results
    
    def aggregate_construction_features_optimized(self, construction_df, grid_df, base_date):
        """최적화된 공사 특성 집계"""
        if len(construction_df) == 0:
            return self._create_empty_construction_features(grid_df)
        
        # 그룹별 집계 (벡터화)
        grouped = construction_df.groupby('grid_id').agg({
            '공사상태': lambda x: [
                sum(status in ['진행중', '시작전'] for status in x),  # active
                sum(status == '완료' for status in x)  # completed
            ],
            '공사유형': lambda x: [
                sum('도로' in str(type_) or '포장' in str(type_) for type_ in x if pd.notna(type_)),
                sum(any(keyword in str(type_) for keyword in ['전력', '상수', '하수', '가스']) 
                    for type_ in x if pd.notna(type_)),
                len(x)
            ],
            '현재_위험도_레벨': 'mean',
            '착공일': 'count',
            '준공일': lambda x: [
                (row['준공일'] - row['착공일']).days 
                for _, row in construction_df[construction_df['grid_id'] == x.name].iterrows()
                if pd.notna(row['착공일']) and pd.notna(row['준공일'])
            ]
        }).reset_index()
        
        # 특성 생성
        features = []
        for _, group in grouped.iterrows():
            grid_id = group['grid_id']
            active_count, completed_count = group['공사상태']
            road_count, utility_count, total_count = group['공사유형']
            duration_list = group['준공일']
            
            features.append({
                'grid_id': grid_id,
                'active_construction_count': active_count,
                'completed_construction_count': completed_count,
                'construction_density': total_count / (self.grid_size_km ** 2),
                'avg_construction_risk': group['현재_위험도_레벨'],
                'road_construction_ratio': road_count / total_count if total_count > 0 else 0,
                'utility_construction_ratio': utility_count / total_count if total_count > 0 else 0,
                'construction_duration_avg': np.mean(duration_list) if duration_list else 0
            })
        
        # 모든 격자에 대해 병합 (없는 격자는 0으로 채움)
        feature_df = pd.DataFrame(features)
        result_df = grid_df[['grid_id']].merge(feature_df, on='grid_id', how='left').fillna(0)
        
        return result_df
    
    def aggregate_weather_features_optimized(self, rainfall_df, grid_df, base_date):
        """최적화된 기상 특성 집계 (성능 대폭 개선)"""
        print(f"   기상 특성 집계 시작... (격자 수: {len(grid_df)})")
        
        # 구별 중심좌표
        district_coords = {
            '강남구': (37.5173, 127.0473), '강동구': (37.5301, 127.1238),
            '강북구': (37.6396, 127.0255), '강서구': (37.5509, 126.8495),
            '관악구': (37.4781, 126.9515), '광진구': (37.5515, 127.0838),
            '구로구': (37.484, 126.8254), '금천구': (37.4563, 126.8956),
            '노원구': (37.6542, 127.0568), '도봉구': (37.6688, 127.0471),
            '동대문구': (37.5838, 127.0507), '동작구': (37.4989, 126.9393),
            '마포구': (37.5663, 126.9019), '서대문구': (37.5791, 126.9368),
            '서초구': (37.4735, 127.0327), '성동구': (37.5633, 127.036),
            '성북구': (37.6023, 127.0188), '송파구': (37.5048, 127.1144),
            '양천구': (37.5269, 126.8555), '영등포구': (37.5263, 126.8963),
            '용산구': (37.5311, 126.9795), '은평구': (37.6176, 126.9227),
            '종로구': (37.5943, 126.9778), '중구': (37.5579, 126.9941),
            '중랑구': (37.6063, 127.0925)
        }
        
        # 날짜 리스트 미리 생성 (30일치)
        target_dates = [base_date - timedelta(days=i) for i in range(30)]
        
        # 구별 강수량 데이터 사전 처리 및 캐싱
        print("   구별 강수량 데이터 캐싱 중...")
        district_rainfall_summary = {}
        
        for district in district_coords.keys():
            district_data = rainfall_df[rainfall_df['지역구'] == district].copy()
            
            # 날짜별 강수량 딕셔너리로 변환 (빠른 조회)
            rainfall_dict = {}
            if not district_data.empty:
                for _, row in district_data.iterrows():
                    rainfall_dict[row['날짜']] = row['누적강수량']
            
            # 미리 계산된 기간별 강수량
            rainfall_1day = rainfall_dict.get(base_date, 0)
            rainfall_3day = sum(rainfall_dict.get(date, 0) for date in target_dates[:3])
            rainfall_7day = sum(rainfall_dict.get(date, 0) for date in target_dates[:7])
            rainfall_30day = sum(rainfall_dict.get(date, 0) for date in target_dates[:30])
            heavy_rain_days = sum(1 for date in target_dates if rainfall_dict.get(date, 0) >= 50)
            
            district_rainfall_summary[district] = {
                'rainfall_1day': rainfall_1day,
                'rainfall_3day': rainfall_3day,
                'rainfall_7day': rainfall_7day,
                'rainfall_30day': rainfall_30day,
                'heavy_rain_days': heavy_rain_days
            }
        
        print("   격자별 가장 가까운 구 계산 중...")
        
        # 벡터화된 거리 계산으로 각 격자의 가장 가까운 구 미리 계산
        district_names = list(district_coords.keys())
        district_coords_array = np.array(list(district_coords.values()))
        
        def find_closest_district_vectorized(grid_coords):
            """벡터화된 가장 가까운 구 찾기"""
            distances = np.sqrt(
                (district_coords_array[:, 0] - grid_coords[0]) ** 2 + 
                (district_coords_array[:, 1] - grid_coords[1]) ** 2
            )
            return district_names[np.argmin(distances)]
        
        # 격자 청크로 나누어 병렬 처리
        chunk_size = max(1, len(grid_df) // (self.n_threads * 2))  # 더 작은 청크
        grid_chunks = [grid_df.iloc[i:i+chunk_size] for i in range(0, len(grid_df), chunk_size)]
        
        print(f"   {len(grid_chunks)}개 청크로 병렬 처리 중...")
        
        def process_weather_chunk_fast(chunk_data):
            """고속화된 기상 청크 처리"""
            chunk_idx, chunk = chunk_data
            chunk_features = []
            
            for _, grid in chunk.iterrows():
                grid_coords = (grid['center_lat'], grid['center_lng'])
                
                # 가장 가까운 구 찾기 (벡터화)
                closest_district = find_closest_district_vectorized(grid_coords)
                
                # 미리 계산된 강수량 데이터 사용
                rainfall_data = district_rainfall_summary.get(closest_district, {
                    'rainfall_1day': 0, 'rainfall_3day': 0, 'rainfall_7day': 0,
                    'rainfall_30day': 0, 'heavy_rain_days': 0
                })
                
                feature = {
                    'grid_id': grid['grid_id'],
                    'closest_district': closest_district,
                    **rainfall_data
                }
                
                chunk_features.append(feature)
            
            # 진행상황 출력
            if chunk_idx % max(1, len(grid_chunks) // 10) == 0:
                progress = (chunk_idx + 1) / len(grid_chunks) * 100
                print(f"     진행률: {progress:.1f}% ({chunk_idx + 1}/{len(grid_chunks)} 청크)")
            
            return chunk_features
        
        # ThreadPoolExecutor로 병렬 실행
        with ThreadPoolExecutor(max_workers=self.n_threads) as executor:
            # 청크에 인덱스 추가
            indexed_chunks = [(i, chunk) for i, chunk in enumerate(grid_chunks)]
            
            futures = [executor.submit(process_weather_chunk_fast, chunk_data) 
                      for chunk_data in indexed_chunks]
            
            all_features = []
            for future in as_completed(futures):
                all_features.extend(future.result())
        
        print(f"   기상 특성 집계 완료: {len(all_features)}개 격자")
        return pd.DataFrame(all_features)
    
    def aggregate_incident_features_optimized(self, pothole_df, grid_df):
        """최적화된 사고 이력 특성 집계 (대폭 성능 개선)"""
        print(f"   사고 이력 특성 집계 시작... (격자 수: {len(grid_df)}, 사고 수: {len(pothole_df)})")
        
        if len(pothole_df) == 0:
            return self._create_empty_incident_features(grid_df)
        
        # 사고 데이터를 NumPy 배열로 변환 (성능 향상)
        incident_coords = pothole_df[['위도', '경도']].values
        incident_damages = pothole_df['피해규모점수'].values
        incident_causes = pothole_df['발생원인구분'].fillna('기타').values
        
        # 격자 좌표도 NumPy 배열로 변환
        grid_coords = grid_df[['center_lat', 'center_lng']].values
        grid_ids = grid_df['grid_id'].values
        
        # 벡터화된 거리 계산 함수
        def calculate_distances_vectorized(grid_coord, incident_coords):
            """벡터화된 거리 계산 (geodesic 대신 유클리드 거리 근사 사용)"""
            # 위도/경도를 km로 변환하는 근사치 (서울 기준)
            lat_diff = (incident_coords[:, 0] - grid_coord[0]) * 111  # 위도 1도 ≈ 111km
            lng_diff = (incident_coords[:, 1] - grid_coord[1]) * 88   # 경도 1도 ≈ 88km (서울 기준)
            distances = np.sqrt(lat_diff**2 + lng_diff**2)
            return distances
        
        # 격자를 청크로 나누어 병렬 처리
        chunk_size = max(1, len(grid_df) // (self.n_threads * 2))
        grid_chunks = [
            (grid_coords[i:i+chunk_size], grid_ids[i:i+chunk_size], i)
            for i in range(0, len(grid_df), chunk_size)
        ]
        
        print(f"   {len(grid_chunks)}개 청크로 병렬 처리 중...")
        
        def process_incident_chunk_fast(chunk_data):
            """고속화된 사고 이력 청크 처리"""
            chunk_coords, chunk_ids, chunk_idx = chunk_data
            chunk_features = []
            
            for i, (grid_coord, grid_id) in enumerate(zip(chunk_coords, chunk_ids)):
                # 벡터화된 거리 계산
                distances = calculate_distances_vectorized(grid_coord, incident_coords)
                
                # 거리별 필터링
                mask_500m = distances <= 0.5
                mask_1km = distances <= 1.0
                
                incidents_500m_count = np.sum(mask_500m)
                incidents_1km_count = np.sum(mask_1km)
                
                # 1km 내 사고 데이터
                if incidents_1km_count > 0:
                    damages_1km = incident_damages[mask_1km]
                    causes_1km = incident_causes[mask_1km]
                    
                    avg_damage = np.mean(damages_1km)
                    max_damage = np.max(damages_1km)
                    cause_diversity = len(set(causes_1km))
                else:
                    avg_damage = 0
                    max_damage = 0
                    cause_diversity = 0
                
                feature = {
                    'grid_id': grid_id,
                    'incident_count_500m': incidents_500m_count,
                    'incident_count_1km': incidents_1km_count,
                    'avg_damage_score': avg_damage,
                    'max_damage_score': max_damage,
                    'incident_cause_diversity': cause_diversity
                }
                
                chunk_features.append(feature)
            
            # 진행상황 출력
            if chunk_idx % max(1, len(grid_chunks) // 10) == 0:
                progress = (chunk_idx + 1) / len(grid_chunks) * 100
                print(f"     진행률: {progress:.1f}% ({chunk_idx + 1}/{len(grid_chunks)} 청크)")
            
            return chunk_features
        
        # ThreadPoolExecutor로 병렬 실행
        with ThreadPoolExecutor(max_workers=self.n_threads) as executor:
            futures = [executor.submit(process_incident_chunk_fast, chunk_data) 
                      for chunk_data in grid_chunks]
            
            all_features = []
            for future in as_completed(futures):
                all_features.extend(future.result())
        
        print(f"   사고 이력 특성 집계 완료: {len(all_features)}개 격자")
        return pd.DataFrame(all_features)
    
    def aggregate_population_features_optimized(self, data_tuple, grid_df):
        """최적화된 인구 밀도 특성 집계"""
        population_df, subway_df = data_tuple
        
        # 격자별 그룹화
        pop_grouped = population_df.groupby('grid_id').agg({
            '방문자수': ['sum', 'std', 'min', 'max']
        }).reset_index() if len(population_df) > 0 else pd.DataFrame()
        
        subway_grouped = subway_df.groupby('grid_id').agg({
            '승하차총승객수': 'sum'
        }).reset_index() if len(subway_df) > 0 else pd.DataFrame()
        
        # 특성 생성
        features = []
        for _, grid in grid_df.iterrows():
            grid_id = grid['grid_id']
            
            # 유동인구 특성
            pop_data = pop_grouped[pop_grouped['grid_id'] == grid_id]
            if len(pop_data) > 0:
                visitor_sum = pop_data[('방문자수', 'sum')].iloc[0]
                visitor_std = pop_data[('방문자수', 'std')].iloc[0]
                visitor_min = pop_data[('방문자수', 'min')].iloc[0]
                visitor_max = pop_data[('방문자수', 'max')].iloc[0]
                
                visitor_density = visitor_sum / (self.grid_size_km ** 2)
                population_variation = visitor_std if pd.notna(visitor_std) else 0
                peak_ratio = visitor_max / (visitor_min + 1) if visitor_min > 0 else 1
            else:
                visitor_density = 0
                population_variation = 0
                peak_ratio = 1
            
            # 지하철 특성
            subway_data = subway_grouped[subway_grouped['grid_id'] == grid_id]
            if len(subway_data) > 0:
                subway_density = subway_data['승하차총승객수'].iloc[0] / (self.grid_size_km ** 2)
            else:
                subway_density = 0
            
            features.append({
                'grid_id': grid_id,
                'visitor_density': visitor_density,
                'subway_passenger_density': subway_density,
                'population_variation': population_variation,
                'peak_population_ratio': peak_ratio
            })
        
        return pd.DataFrame(features)
    
    def create_target_variable_optimized(self, data_tuple, grid_df):
        """최적화된 타겟 변수 생성"""
        construction_df, pothole_df = data_tuple
        
        # 격자별 그룹화
        construction_grouped = construction_df.groupby('grid_id').agg({
            '현재_위험도_레벨': 'mean',
            '공사상태': lambda x: sum(status in ['진행중', '시작전'] for status in x)
        }).reset_index() if len(construction_df) > 0 else pd.DataFrame()
        
        incident_grouped = pothole_df.groupby('grid_id').agg({
            '피해규모점수': 'max'
        }).reset_index() if len(pothole_df) > 0 else pd.DataFrame()
        
        targets = []
        for _, grid in grid_df.iterrows():
            grid_id = grid['grid_id']
            
            # 사고 발생 여부
            has_incident = 1 if grid_id in incident_grouped['grid_id'].values else 0
            
            # 종합 위험도 계산
            risk_score = 0
            
            # 공사 위험도 (40%)
            construction_data = construction_grouped[construction_grouped['grid_id'] == grid_id]
            if len(construction_data) > 0:
                construction_risk = construction_data['현재_위험도_레벨'].iloc[0]
                active_count = construction_data['공사상태'].iloc[0]
                risk_score += construction_risk * 0.4
                risk_score += min(active_count, 4) * 0.2
            
            # 과거 사고 이력 (40%)
            incident_data = incident_grouped[incident_grouped['grid_id'] == grid_id]
            if len(incident_data) > 0:
                incident_risk = incident_data['피해규모점수'].iloc[0]
                risk_score += incident_risk * 0.4
            
            # 위험도 레벨로 변환 (0-4)
            if risk_score == 0:
                risk_level = 0
            elif risk_score <= 1:
                risk_level = 1
            elif risk_score <= 2:
                risk_level = 2
            elif risk_score <= 3:
                risk_level = 3
            else:
                risk_level = 4
            
            targets.append({
                'grid_id': grid_id,
                'risk_level': risk_level,
                'has_incident': has_incident,
                'risk_score': round(risk_score, 3)
            })
        
        return pd.DataFrame(targets)
    
    def _create_empty_construction_features(self, grid_df):
        """빈 공사 특성 데이터프레임 생성"""
        empty_features = []
        for _, grid in grid_df.iterrows():
            empty_features.append({
                'grid_id': grid['grid_id'],
                'active_construction_count': 0,
                'completed_construction_count': 0,
                'construction_density': 0,
                'avg_construction_risk': 0,
                'road_construction_ratio': 0,
                'utility_construction_ratio': 0,
                'construction_duration_avg': 0
            })
        return pd.DataFrame(empty_features)
    
    def _create_empty_incident_features(self, grid_df):
        """빈 사고 이력 특성 데이터프레임 생성"""
        empty_features = []
        for _, grid in grid_df.iterrows():
            empty_features.append({
                'grid_id': grid['grid_id'],
                'incident_count_500m': 0,
                'incident_count_1km': 0,
                'avg_damage_score': 0,
                'max_damage_score': 0,
                'incident_cause_diversity': 0
            })
        return pd.DataFrame(empty_features)
    
    def process_all_data_optimized(self, base_date=None):
        """병렬 처리 최적화된 전체 데이터 처리 파이프라인"""
        if base_date is None:
            base_date = datetime(2024, 6, 25)
        
        total_start_time = time.time()
        print("=== 병렬 처리 최적화 싱크홀 위험도 예측 데이터 전처리 시작 ===")
        
        # 1. 병렬 데이터 로드
        print(f"\n1. 병렬 CSV 데이터 로드 중... (스레드: {self.n_threads}개)")
        load_start = time.time()
        dataframes = self.load_csv_data_parallel()
        load_time = time.time() - load_start
        print(f"   로드 완료 시간: {load_time:.2f}초")
        
        # 2. 병렬 데이터 정제
        print(f"\n2. 병렬 데이터 정제 중... (스레드: {min(5, self.n_threads)}개)")
        clean_start = time.time()
        cleaned_dataframes = self.clean_data_parallel(dataframes)
        clean_time = time.time() - clean_start
        print(f"   정제 완료 시간: {clean_time:.2f}초")
        
        # 3. 격자 시스템 생성 (벡터화)
        print(f"\n3. 벡터화 격자 시스템 생성 중...")
        grid_start = time.time()
        grid_df = self.create_seoul_grid()
        grid_time = time.time() - grid_start
        print(f"   격자 생성 시간: {grid_time:.2f}초")
        
        # 4. 병렬 격자 할당
        print(f"\n4. 병렬 격자 할당 중... (벡터화 최적화)")
        assign_start = time.time()
        assigned_dataframes = self.assign_all_to_grid_parallel(cleaned_dataframes, grid_df)
        assign_time = time.time() - assign_start
        print(f"   격자 할당 시간: {assign_time:.2f}초")
        
        # 5. 병렬 특성 집계
        print(f"\n5. 병렬 특성 집계 중... (스레드: {min(5, self.n_threads)}개)")
        agg_start = time.time()
        feature_results = self.aggregate_features_parallel(assigned_dataframes, grid_df, base_date)
        agg_time = time.time() - agg_start
        print(f"   특성 집계 시간: {agg_time:.2f}초")
        
        # 6. 최종 데이터셋 통합
        print(f"\n6. 최종 데이터셋 통합 중...")
        merge_start = time.time()
        final_df = grid_df[['grid_id', 'center_lat', 'center_lng']].copy()
        
        # 모든 특성 병합
        for feature_name, feature_df in feature_results.items():
            final_df = final_df.merge(feature_df, on='grid_id', how='left')
        
        # 시간적 특성 추가
        final_df['date'] = base_date
        final_df['month'] = base_date.month
        final_df['quarter'] = (base_date.month - 1) // 3 + 1
        final_df['is_rainy_season'] = 1 if base_date.month in [6, 7, 8] else 0
        final_df['day_of_year'] = base_date.timetuple().tm_yday
        
        # 결측치 처리
        final_df = final_df.fillna(0)
        
        # 병렬로 파생 특성 생성
        with ThreadPoolExecutor(max_workers=self.n_threads) as executor:
            futures = []
            
            # 상호작용 특성들을 병렬로 계산
            futures.append(executor.submit(
                lambda: final_df['construction_density'] * final_df['rainfall_7day']
            ))
            futures.append(executor.submit(
                lambda: final_df['visitor_density'] / (final_df['construction_density'] + 1)
            ))
            futures.append(executor.submit(
                lambda: (final_df['avg_construction_risk'] * 0.3 + 
                        final_df['max_damage_score'] * 0.4 + 
                        final_df['heavy_rain_days'] * 0.3)
            ))
            
            # 결과 수집
            final_df['construction_weather_interaction'] = futures[0].result()
            final_df['population_construction_ratio'] = futures[1].result()
            final_df['vulnerability_index'] = futures[2].result()
        
        merge_time = time.time() - merge_start
        print(f"   데이터 통합 시간: {merge_time:.2f}초")
        
        total_time = time.time() - total_start_time
        
        print(f"\n=== 병렬 처리 전처리 완료 ===")
        print(f"전체 처리 시간: {total_time:.2f}초")
        print(f"최종 데이터셋 크기: {final_df.shape}")
        print(f"특성 수: {final_df.shape[1] - 4}")  # grid_id, lat, lng, date 제외
        print(f"위험도 분포:")
        print(final_df['risk_level'].value_counts().sort_index())
        
        # 성능 요약
        print(f"\n=== 성능 요약 ===")
        print(f"데이터 로드: {load_time:.2f}초")
        print(f"데이터 정제: {clean_time:.2f}초")  
        print(f"격자 생성: {grid_time:.2f}초")
        print(f"격자 할당: {assign_time:.2f}초")
        print(f"특성 집계: {agg_time:.2f}초")
        print(f"데이터 통합: {merge_time:.2f}초")
        print(f"총 처리 시간: {total_time:.2f}초")
        print(f"병렬 효율성: {self.n_processes}x 프로세스, {self.n_threads}x 스레드")
        
        return final_df
    
    def save_processed_data_parallel(self, final_df, output_path='sinkhole_processed_data.csv'):
        """병렬로 처리된 데이터 저장"""
        def save_csv():
            final_df.to_csv(output_path, index=False, encoding='utf-8-sig')
            return f"CSV 파일이 {output_path}에 저장되었습니다."
        
        def save_summary():
            summary_info = {
                'total_grids': len(final_df),
                'feature_count': final_df.shape[1] - 4,
                'risk_distribution': final_df['risk_level'].value_counts().to_dict(),
                'incident_rate': final_df['has_incident'].mean(),
                'date_processed': datetime.now().isoformat(),
                'processing_config': {
                    'grid_size_km': self.grid_size_km,
                    'n_processes': self.n_processes,
                    'n_threads': self.n_threads
                }
            }
            
            import json
            with open('data_summary.json', 'w', encoding='utf-8') as f:
                json.dump(summary_info, f, ensure_ascii=False, indent=2)
            return "요약 정보가 data_summary.json에 저장되었습니다."
        
        # 병렬로 파일 저장
        with ThreadPoolExecutor(max_workers=2) as executor:
            csv_future = executor.submit(save_csv)
            summary_future = executor.submit(save_summary)
            
            print(csv_future.result())
            print(summary_future.result())

# 성능 비교 및 벤치마크 클래스
class PerformanceBenchmark:
    """성능 비교 및 벤치마크"""
    
    @staticmethod
    def compare_performance(data_size_multiplier=1):
        """기존 방식과 병렬 처리 방식 성능 비교"""
        print("=== 성능 비교 벤치마크 ===")
        
        # 병렬 처리 방식
        print(f"\n1. 병렬 처리 방식 (데이터 크기: {data_size_multiplier}x)")
        parallel_processor = OptimizedSinkholeDataPreprocessor(
            grid_size_km=0.5,
            n_processes=max(1, cpu_count() - 1),
            n_threads=min(32, (cpu_count() or 1) + 4)
        )
        
        parallel_start = time.time()
        try:
            parallel_result = parallel_processor.process_all_data_optimized()
            parallel_time = time.time() - parallel_start
            print(f"병렬 처리 완료 시간: {parallel_time:.2f}초")
        except Exception as e:
            print(f"병렬 처리 중 오류: {e}")
            parallel_time = float('inf')
        
        print(f"\n=== 최종 성능 요약 ===")
        print(f"병렬 처리 시간: {parallel_time:.2f}초")
        print(f"사용된 프로세스: {parallel_processor.n_processes}개")
        print(f"사용된 스레드: {parallel_processor.n_threads}개")
        print(f"예상 성능 향상: {parallel_processor.n_processes:.1f}x ~ {parallel_processor.n_processes * 2:.1f}x")

# 실행 예시 및 사용법
if __name__ == "__main__":
    # 기본 사용법
    print("=== 병렬 처리 최적화 전처리 실행 ===")
    
    # 1. 최적화된 전처리 실행
    processor = OptimizedSinkholeDataPreprocessor(
        grid_size_km=0.5,  # 500m x 500m 격자
        n_processes=None,  # 자동 설정 (CPU 코어 수 - 1)
        n_threads=None     # 자동 설정
    )
    
    try:
        # 전체 데이터 처리 (병렬 최적화)
        final_dataset = processor.process_all_data_optimized(
            base_date=datetime(2024, 6, 25)
        )
        
        # 결과 저장 (병렬)
        processor.save_processed_data_parallel(final_dataset)
        
        # 기본 통계 출력
        print("\n=== 최종 데이터 통계 ===")
        print("수치형 특성 요약:")
        numeric_cols = final_dataset.select_dtypes(include=[np.number]).columns
        print(final_dataset[numeric_cols].describe())
        
        print("\n범주형 특성 요약:")
        categorical_cols = final_dataset.select_dtypes(exclude=[np.number]).columns
        for col in categorical_cols:
            if col not in ['grid_id', 'date']:
                print(f"{col}: {final_dataset[col].value_counts().to_dict()}")
        
        # 성능 벤치마크 (선택사항)
        print("\n=== 성능 벤치마크 실행 ===")
        PerformanceBenchmark.compare_performance()
        
    except FileNotFoundError as e:
        print(f"파일을 찾을 수 없습니다: {e}")
        print("CSV 파일들이 현재 디렉토리에 있는지 확인해주세요:")
        print("- 공사현황정제.csv")
        print("- 지역구별누적강수량.csv") 
        print("- 포트홀및싱크홀정보.csv")
        print("- 유동인구수.csv")
        print("- 지하철유동인구수.csv")
        
    except Exception as e:
        print(f"처리 중 오류 발생: {e}")
        print("데이터 형식과 컬럼명을 확인해주세요.")

# 추가 최적화 팁
"""
=== 추가 성능 최적화 방법 ===

1. 메모리 최적화:
   - 대용량 데이터의 경우 청크 단위 처리
   - 불필요한 컬럼 조기 제거
   - 데이터 타입 최적화 (int64 → int32, float64 → float32)

2. I/O 최적화:
   - CSV 대신 Parquet 파일 사용 고려
   - 압축된 파일 포맷 사용
   - SSD 사용 권장

3. 알고리즘 최적화:
   - 거리 계산 시 KDTree 사용 고려
   - 공간 인덱싱 (R-tree) 활용
   - 캐싱 전략 적용

4. 하드웨어 최적화:
   - 더 많은 CPU 코어 활용
   - 충분한 RAM 확보 (최소 8GB, 권장 16GB+)
   - 네트워크 스토리지보다 로컬 스토리지 사용

5. Azure ML 최적화:
   - 컴퓨팅 인스턴스 사이즈 업그레이드
   - 병렬 파이프라인 구성
   - 데이터스토어 캐싱 활용
"""

=== 병렬 처리 최적화 전처리 실행 ===
병렬 처리 설정: 11개 프로세스, 16개 스레드
=== 병렬 처리 최적화 싱크홀 위험도 예측 데이터 전처리 시작 ===

1. 병렬 CSV 데이터 로드 중... (스레드: 16개)
병렬 CSV 로드 시작...
✓ construction 데이터 로드 완료: 389건 (0.01초)
✓ population 데이터 로드 완료: 18880건 (0.01초)
✓ pothole 데이터 로드 완료: 20470건 (0.04초)
✓ rainfall 데이터 로드 완료: 64925건 (0.05초)
✓ subway 데이터 로드 완료: 741755건 (0.35초)
   로드 완료 시간: 0.36초

2. 병렬 데이터 정제 중... (스레드: 5개)
병렬 데이터 정제 시작...
✓ construction 데이터 정제 완료: 389건 (0.01초)
✓ population 데이터 정제 완료: 18880건 (0.02초)
✓ rainfall 데이터 정제 완료: 64925건 (0.03초)
✓ pothole 데이터 정제 완료: 20451건 (0.14초)
✓ subway 데이터 정제 완료: 741755건 (0.19초)
   정제 완료 시간: 0.20초

3. 벡터화 격자 시스템 생성 중...
격자 시스템 생성 완료: 4757개 격자
   격자 생성 시간: 0.00초

4. 병렬 격자 할당 중... (벡터화 최적화)
병렬 격자 할당 시작...
✓ construction 격자 할당 완료: 389건 (0.00초)
✓ population 격자 할당 완료: 18068건 (0.03초)
✓ pothole 격자 할당 완료: 20451건 (0.04초)
✓ subway 격자 할당 완료: 731903건 (0.58초)
   격자 할당 시간: 0.59초

5. 병렬 특성 집계 중... (스레드: 5개)
병렬 특성 집계 시작...
   공사 특성 집계 시작...
   기상 특성 집계 시작... (격자 수: 4757)
   구별 강수량 데이터 캐싱 중...
   사고 이력 특성 집

'\n=== 추가 성능 최적화 방법 ===\n\n1. 메모리 최적화:\n   - 대용량 데이터의 경우 청크 단위 처리\n   - 불필요한 컬럼 조기 제거\n   - 데이터 타입 최적화 (int64 → int32, float64 → float32)\n\n2. I/O 최적화:\n   - CSV 대신 Parquet 파일 사용 고려\n   - 압축된 파일 포맷 사용\n   - SSD 사용 권장\n\n3. 알고리즘 최적화:\n   - 거리 계산 시 KDTree 사용 고려\n   - 공간 인덱싱 (R-tree) 활용\n   - 캐싱 전략 적용\n\n4. 하드웨어 최적화:\n   - 더 많은 CPU 코어 활용\n   - 충분한 RAM 확보 (최소 8GB, 권장 16GB+)\n   - 네트워크 스토리지보다 로컬 스토리지 사용\n\n5. Azure ML 최적화:\n   - 컴퓨팅 인스턴스 사이즈 업그레이드\n   - 병렬 파이프라인 구성\n   - 데이터스토어 캐싱 활용\n'